# Age Classifier — MobileNetV2 v2 (Mejorado)
**Tiempo estimado: ~15-20 min en T4 GPU**

### Problemas corregidos respecto a v1
| Problema | Causa | Solución |
|---|---|---|
| Mala predicción en teens 13-17 | Solo 670 imgs vs 1112 de bebés | Caps por grupo de edad + oversample ×3 |
| Bug train/inferencia | Training sin padding, inferencia con padding cuadrado | Mismo preprocesado en ambos |
| Doble compensación inestable | Focal Loss + class_weight{1:4.0} simultáneos | Solo BCE + label_smoothing + class_weight{1:2.0} |
| Fine-tuning brusco | 2 fases, descongelado de golpe | 3 fases con descongelado gradual |
| Umbral fijo 0.5 | No adaptado al problema | Búsqueda por F2-score (prioriza recall de menores) |

In [ ]:
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    print('SIN GPU — Ve a: Runtime > Change runtime type > T4 GPU')
else:
    print(f'GPU: {gpus[0].name}')

In [ ]:
# Actualizar kaggle PRIMERO — Colab trae version antigua que no soporta tokens nuevos de Kaggle
!pip install -q --upgrade kaggle
!pip install -q scikit-learn matplotlib opencv-python-headless scipy

## Paso 1 — Dataset de Kaggle

**Orden obligatorio:**
1. Ejecuta la celda de arriba (`pip install --upgrade kaggle`) y espera a que termine
2. Ve a https://www.kaggle.com/settings → API → **Create New Token** → descarga `kaggle.json`
3. Ejecuta la celda de abajo y sube el fichero cuando aparezca el botón

In [ ]:
from google.colab import files
uploaded = files.upload()  # sube kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d frabbisw/facial-age -p /content --unzip
import os, pathlib

# Detectar la subcarpeta raiz del dataset automaticamente
dataset_root = None
for p in pathlib.Path('/content').iterdir():
    if p.is_dir() and any(p.iterdir()):
        children = list(p.iterdir())
        if any(c.is_dir() and (c.name.lstrip('0') or '0').isdigit() for c in children):
            dataset_root = str(p)
            break
if not dataset_root:
    dataset_root = '/content'

n_folders = len([x for x in pathlib.Path(dataset_root).iterdir() if x.is_dir()])
print(f'Dataset en: {dataset_root}')
print(f'Carpetas de edad: {n_folders}')

## Paso 2 — Analisis del dataset
Inspeccionamos la distribucion real para entender el desbalanceo.

In [ ]:
import pathlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

MINOR_THRESHOLD = 18
DATASET_ROOT = pathlib.Path(dataset_root)

age_data = {}
for d in sorted(DATASET_ROOT.iterdir()):
    if not d.is_dir():
        continue
    name = d.name.lstrip('0') or '0'
    if not name.isdigit():
        continue
    age = int(name)
    imgs = list(d.glob('*.png')) + list(d.glob('*.jpg')) + list(d.glob('*.jpeg'))
    age_data[age] = len(imgs)

ages = sorted(age_data.keys())
counts = [age_data[a] for a in ages]
colors = ['#e74c3c' if a < MINOR_THRESHOLD else '#3498db' for a in ages]

menores_total = sum(c for a, c in age_data.items() if a < MINOR_THRESHOLD)
adultos_total = sum(c for a, c in age_data.items() if a >= MINOR_THRESHOLD)
total = menores_total + adultos_total

print('=' * 55)
print(f'  MENORES (<18):  {menores_total:5d} imgs ({100*menores_total/total:.1f}%)')
print(f'  ADULTOS (>=18): {adultos_total:5d} imgs ({100*adultos_total/total:.1f}%)')
print(f'  TOTAL:          {total:5d} imgs')
print(f'  Ratio adulto/menor: {adultos_total/menores_total:.2f}x')
print('=' * 55)

print(f'\nBEBES (edad 1): {age_data.get(1, 0)} imgs = {100*age_data.get(1,0)/menores_total:.1f}% de los menores')
print('  => Dominan la clase menor, el modelo aprende "menor = bebe"')

print(f'\nEDADES CRITICAS FRONTERA (13-17): {sum(age_data.get(a,0) for a in range(13,18))} imgs total')
for a in range(13, 18):
    print(f'  Edad {a}: {age_data.get(a, 0):4d} imgs')
print('  => Son los MAS DIFICILES y los MENOS representados')

print(f'\nADULTOS JOVENES FRONTERA (18-22):')
for a in range(18, 23):
    print(f'  Edad {a}: {age_data.get(a, 0):4d} imgs')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(ages, counts, color=colors, width=0.8)
axes[0].axvline(x=17.5, color='black', linestyle='--', linewidth=2, label='Frontera 18 anhos')
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Imagenes')
axes[0].set_title('Distribucion del dataset por edad')
legend_els = [Patch(facecolor='#e74c3c', label=f'Menores <18: {menores_total}'),
               Patch(facecolor='#3498db', label=f'Adultos >=18: {adultos_total}'),
               plt.Line2D([0], [0], color='black', ls='--', label='Frontera')]
axes[0].legend(handles=legend_els)

# Zoom en la zona critica 10-25
critical_ages = [a for a in ages if 10 <= a <= 25]
critical_counts = [age_data[a] for a in critical_ages]
critical_colors = ['#e74c3c' if a < MINOR_THRESHOLD else '#3498db' for a in critical_ages]
axes[1].bar(critical_ages, critical_counts, color=critical_colors, width=0.8)
axes[1].axvline(x=17.5, color='black', linestyle='--', linewidth=2)
axes[1].set_xlabel('Edad')
axes[1].set_ylabel('Imagenes')
axes[1].set_title('Zona critica: edades 10-25 (frontera menor/adulto)')
for a, c in zip(critical_ages, critical_counts):
    axes[1].text(a, c + 1, str(c), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('/content/dataset_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print('Grafico guardado: /content/dataset_analysis.png')

## Paso 3 — Configuracion e imports

In [ ]:
import json, logging, os
from pathlib import Path
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, fbeta_score, precision_recall_curve
)
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import callbacks, layers, models, regularizers
from tensorflow.keras.applications import MobileNetV2

DATASET_DIR = Path(dataset_root)
OUTPUT_DIR  = Path('/content/model')
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_PATH  = OUTPUT_DIR / 'age_classifier.keras'

# ── Dimensiones — deben coincidir con age-detection/main.py ──────────────────
IMG_SIZE = (200, 200)

# ── Hiperparametros ──────────────────────────────────────────────────────────
BATCH_SIZE       = 32
EPOCHS_HEAD      = 25   # Fase 1: cabeza, base congelada
EPOCHS_FT1       = 20   # Fase 2: descongelar ultimas 40 capas
EPOCHS_FT2       = 15   # Fase 3: descongelar ultimas 80 capas

LR_HEAD          = 1e-3
LR_FT1           = 2e-5
LR_FT2           = 5e-6

# ── Dataset ──────────────────────────────────────────────────────────────────
MINOR_THRESHOLD     = 18
VAL_SPLIT           = 0.2
SEED                = 42

# Oversampling en la zona fronteriza: son los ejemplos mas dificiles
TEEN_REPEAT         = 3  # Teens 13-17: triplicar (los mas dificiles y escasos)
YOUNG_ADULT_REPEAT  = 2  # Adultos jovenes 18-21: duplicar (hard negatives)

# class_weight sin Focal Loss — compensacion moderada
CLASS_WEIGHT_MINOR  = 2.0  # {0: 1.0, 1: 2.0}

print(f'TF {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

## Paso 4 — Funciones auxiliares

In [ ]:
def get_age_cap(age: int) -> int:
    """
    Cap de imagenes por edad.
    Logica: bebes (triviales) -> pocos; teens (criticos) -> todos; adultos mayores -> pocos.
    """
    if age == 1:  return 15   # Bebes son triviales de clasificar; 15 imagenes bastan
    if age <= 3:  return 60   # Primera infancia
    if age <= 5:  return 80   # Preescolar
    if age <= 12: return 9999 # Infancia media: preservar todo
    if age <= 17: return 9999 # Adolescentes CRITICOS: preservar todo
    if age <= 23: return 150  # Adultos jovenes — frontera dificil, necesitamos muchos
    if age <= 45: return 65   # Adultos — casos mas faciles
    return 20                  # Mayores — muy faciles de clasificar


def load_dataset(dataset_dir: Path):
    paths, labels, ages = [], [], []
    rng = np.random.default_rng(SEED)
    raw_counts = {0: 0, 1: 0}
    capped_counts = {0: 0, 1: 0}

    for age_dir in sorted(dataset_dir.iterdir()):
        if not age_dir.is_dir():
            continue
        name = age_dir.name.lstrip('0') or '0'
        if not name.isdigit():
            continue
        age   = int(name)
        label = 1 if age < MINOR_THRESHOLD else 0
        imgs  = (list(age_dir.glob('*.png')) + list(age_dir.glob('*.jpg'))
                 + list(age_dir.glob('*.jpeg')))
        imgs  = [str(p) for p in imgs]
        raw_counts[label] += len(imgs)

        cap = get_age_cap(age)
        if len(imgs) > cap:
            idx  = rng.choice(len(imgs), cap, replace=False)
            imgs = [imgs[i] for i in idx]

        capped_counts[label] += len(imgs)
        for p in imgs:
            paths.append(p)
            labels.append(label)
            ages.append(age)

    n_total = len(paths)
    print(f'Dataset tras caps por edad:')
    print(f'  Menores: {capped_counts[1]}  |  Adultos: {capped_counts[0]}  |  Total: {n_total}')
    print(f'  Ratio: {capped_counts[0]/capped_counts[1]:.2f}x adultos por menor')
    return np.array(paths), np.array(labels, dtype=np.int32), np.array(ages, dtype=np.int32)


def oversample_boundary(paths, labels, ages):
    """
    Oversampling de la zona fronteriza:
    - Teens 13-17: x3 (son los MAS dificiles y los menos representados)
    - Adultos jovenes 18-21: x2 (hard negatives, ayudan a NO confundirlos con menores)
    Solo se aplica al conjunto de train, ANTES de crear el tf.data.Dataset.
    La augmentacion (flip, brillo, zoom...) se aplica despues en cada epoch,
    por lo que cada copia del mismo archivo recibe distorsiones distintas.
    """
    extra_p, extra_l = [], []
    teen_added = ya_added = 0
    for p, l, a in zip(paths, labels, ages):
        if 13 <= a <= 17:
            for _ in range(TEEN_REPEAT - 1):
                extra_p.append(p)
                extra_l.append(l)
            teen_added += TEEN_REPEAT - 1
        elif 18 <= a <= 21:
            for _ in range(YOUNG_ADULT_REPEAT - 1):
                extra_p.append(p)
                extra_l.append(l)
            ya_added += YOUNG_ADULT_REPEAT - 1

    if extra_p:
        paths  = np.concatenate([paths, extra_p])
        labels = np.concatenate([labels, extra_l])
        print(f'Oversample teens 13-17: +{teen_added} imgs')
        print(f'Oversample adultos jovenes 18-21: +{ya_added} imgs')
        print(f'Total train tras oversample: {len(paths)} imgs')
    return paths, labels


def preprocess(path, label):
    """
    IMPORTANTE: mismo preprocesado que age-detection/main.py.
    La version anterior NO tenia padding cuadrado durante el entrenamiento
    pero si durante la inferencia -> el modelo veia distribuciones distintas.
    """
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.cast(img, tf.float32)

    # Padding cuadrado (igual que en inferencia en age-detection/main.py)
    shape = tf.shape(img)
    h, w  = shape[0], shape[1]
    side  = tf.maximum(h, w)
    pad_t = (side - h) // 2
    pad_b = side - h - pad_t
    pad_l = (side - w) // 2
    pad_r = side - w - pad_l
    img   = tf.pad(img, [[pad_t, pad_b], [pad_l, pad_r], [0, 0]], mode='REFLECT')

    img = tf.image.resize(img, IMG_SIZE)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img, label


def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.25)
    img = tf.image.random_contrast(img, lower=0.75, upper=1.25)
    img = tf.image.random_saturation(img, lower=0.75, upper=1.25)
    img = tf.image.random_hue(img, max_delta=0.06)
    # Zoom aleatorio: ampliar 12% y recortar a tamano original
    zoom_h = int(IMG_SIZE[0] * 1.12)
    zoom_w = int(IMG_SIZE[1] * 1.12)
    img = tf.image.resize(img, [zoom_h, zoom_w])
    img = tf.image.random_crop(img, size=[IMG_SIZE[0], IMG_SIZE[1], 3])
    img = tf.clip_by_value(img, -1.0, 1.0)
    return img, label


def make_dataset(paths, labels, augment_data=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


def build_model(trainable_base=False):
    """
    Head simplificado respecto a v1 (256+128 -> 128 solo).
    Menos parametros en la cabeza = menos overfitting.
    """
    base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
    base.trainable = trainable_base

    inp = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(128, activation='relu',
                        kernel_regularizer=regularizers.l2(1e-4))(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid', name='minor_prob')(x)
    return models.Model(inp, out, name='age_classifier')


def get_callbacks(model_path, monitor='val_auc', mode='max'):
    return [
        callbacks.EarlyStopping(
            monitor=monitor, patience=7,
            restore_best_weights=True, mode=mode, verbose=1
        ),
        callbacks.ModelCheckpoint(
            str(model_path), monitor=monitor,
            save_best_only=True, mode=mode, verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.4,
            patience=3, min_lr=1e-7, verbose=1
        ),
    ]


# ── Configuracion de perdida y metricas ──────────────────────────────────────
# BCE con label_smoothing=0.05 — suaviza targets 0/1 a 0.025/0.975
# Previene sobreconfianza sin necesitar Focal Loss adicional.
LOSS    = tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05)
METRICS = [
    'accuracy',
    tf.keras.metrics.AUC(name='auc'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.Precision(name='precision'),
]
WGTS = {0: 1.0, 1: CLASS_WEIGHT_MINOR}

print('Funciones cargadas correctamente.')

## Paso 5 — Cargar y equilibrar el dataset

In [ ]:
all_p, all_l, all_a = load_dataset(DATASET_DIR)

# Split estratificado
tr_p, val_p, tr_l, val_l, tr_a, val_a = train_test_split(
    all_p, all_l, all_a,
    test_size=VAL_SPLIT, stratify=all_l, random_state=SEED
)
n_minor_tr = tr_l.sum()
n_adult_tr = len(tr_l) - n_minor_tr
print(f'\nTrain base:  {len(tr_p)}  (menores={n_minor_tr}, adultos={n_adult_tr})')
print(f'Validacion:  {len(val_p)}  (menores={val_l.sum()}, adultos={len(val_l)-val_l.sum()})')

# Oversampling de zonas fronterizas (solo en train)
print('\nAplicando oversampling frontera...')
tr_p, tr_l = oversample_boundary(tr_p, tr_l, tr_a)

train_ds = make_dataset(tr_p, tr_l, augment_data=True,  shuffle=True)
val_ds   = make_dataset(val_p, val_l, augment_data=False, shuffle=False)

print(f'\nClass weights: {WGTS}')
print(f'Loss: BCE con label_smoothing=0.05')
print(f'Nota: NO se usa Focal Loss (evitar doble compensacion con class_weight)')

## Paso 6 — Entrenamiento
### Fase 1 — Cabeza solamente (base MobileNetV2 congelada)

In [ ]:
print('=' * 60)
print('FASE 1 — Cabeza (base congelada)')
print('=' * 60)

model = build_model(trainable_base=False)
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_HEAD),
    loss=LOSS,
    metrics=METRICS
)
model.summary(line_length=80)

h1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=WGTS,
    callbacks=get_callbacks(MODEL_PATH),
    verbose=1,
)

print(f'\nFase 1 completada — epocas ejecutadas: {len(h1.history["loss"])}')
print(f'Mejor val_AUC fase 1: {max(h1.history["val_auc"]):.4f}')

### Fase 2 — Fine-tuning ultimas 40 capas

In [ ]:
print('=' * 60)
print('FASE 2 — Fine-tuning ultimas 40 capas')
print('=' * 60)

base = model.layers[1]  # MobileNetV2
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

n_trainable = sum(1 for l in model.trainable_variables)
print(f'Variables entrenables: {n_trainable}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FT1),
    loss=LOSS,
    metrics=METRICS
)

h2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT1,
    class_weight=WGTS,
    callbacks=get_callbacks(MODEL_PATH),
    verbose=1,
)

print(f'\nFase 2 completada — epocas: {len(h2.history["loss"])}')
print(f'Mejor val_AUC fase 2: {max(h2.history["val_auc"]):.4f}')

### Fase 3 — Fine-tuning ultimas 80 capas (descongelado profundo)

In [ ]:
print('=' * 60)
print('FASE 3 — Fine-tuning ultimas 80 capas')
print('=' * 60)

base = model.layers[1]
base.trainable = True
for layer in base.layers[:-80]:
    layer.trainable = False

n_trainable = sum(1 for l in model.trainable_variables)
print(f'Variables entrenables: {n_trainable}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_FT2),
    loss=LOSS,
    metrics=METRICS
)

h3 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT2,
    class_weight=WGTS,
    callbacks=get_callbacks(MODEL_PATH),
    verbose=1,
)

# Guardar si ModelCheckpoint no lo hizo (por si val_auc no mejoro en ft3)
if not MODEL_PATH.exists():
    model.save(MODEL_PATH)

print(f'\nFase 3 completada — epocas: {len(h3.history["loss"])}')
print(f'Mejor val_AUC fase 3: {max(h3.history["val_auc"]):.4f}')
print(f'Modelo guardado en: {MODEL_PATH}')

## Paso 7 — Evaluacion y seleccion de umbral optimo

In [ ]:
best_model = tf.keras.models.load_model(MODEL_PATH)
probs = best_model.predict(val_ds, verbose=0).ravel()

auc = roc_auc_score(val_l, probs)
print(f'AUC-ROC (validacion): {auc:.4f}\n')

# ── Busqueda del umbral optimo por F2-score ───────────────────────────────────
# F2 pondera el recall 2x mas que la precision: prioriza NO dejar pasar menores.
# Restriccion: precision minima de 0.50 para no pixelar todos los adultos.
thresholds = np.arange(0.20, 0.75, 0.025)
best_thr   = 0.5
best_f2    = 0.0

print('Busqueda de umbral optimo (max F2-score, precision >= 0.50):')
print(f'{"Thr":>6} {"Precision":>10} {"Recall":>8} {"F2":>8}')
for thr in thresholds:
    preds = (probs >= thr).astype(int)
    report = classification_report(val_l, preds, output_dict=True, zero_division=0)
    minor  = report.get('1', {})
    p  = minor.get('precision', 0.0)
    r  = minor.get('recall', 0.0)
    f2 = fbeta_score(val_l, preds, beta=2, pos_label=1, zero_division=0)
    flag = '  <-- OPTIMO' if (f2 > best_f2 and p >= 0.50) else ''
    print(f'{thr:6.3f} {p:10.3f} {r:8.3f} {f2:8.3f}{flag}')
    if f2 > best_f2 and p >= 0.50:
        best_f2, best_thr = f2, thr

print(f'\n>>> Umbral optimo: {best_thr:.3f}  (F2={best_f2:.3f})')

# ── Reporte detallado con umbral optimo ───────────────────────────────────────
print('\n' + '=' * 60)
print(f'REPORTE FINAL — Umbral {best_thr:.3f}')
print('=' * 60)
preds_best = (probs >= best_thr).astype(int)
print(classification_report(val_l, preds_best, target_names=['adulto', 'menor'], zero_division=0))
cm = confusion_matrix(val_l, preds_best)
print('Matriz de confusion:')
print(f'               Pred adulto  Pred menor')
print(f'  Real adulto: {cm[0,0]:10d}  {cm[0,1]:10d}')
print(f'  Real menor:  {cm[1,0]:10d}  {cm[1,1]:10d}')
print(f'  Falsos negativos (menores perdidos): {cm[1,0]}')
print(f'  Falsos positivos (adultos pixelados): {cm[0,1]}')

# ── Evaluacion por grupo de edad ───────────────────────────────────────────────
print('\n' + '=' * 60)
print('PRECISION POR GRUPO DE EDAD (val set)')
print('=' * 60)
groups = [
    ('Bebes 1-5',        1,   5),
    ('Infancia 6-12',    6,  12),
    ('Teens 13-17',     13,  17),
    ('Adultos 18-22',   18,  22),
    ('Adultos 23-40',   23,  40),
    ('Adultos 41+',     41, 999),
]
for group_name, lo, hi in groups:
    mask = np.array([(lo <= a <= hi) for a in val_a])
    if mask.sum() == 0:
        continue
    g_preds = (probs[mask] >= best_thr).astype(int)
    g_true  = val_l[mask]
    correct = (g_preds == g_true).sum()
    n = mask.sum()
    # Calcular false negatives para menores
    if g_true.sum() > 0:  # hay menores en este grupo
        fn = ((g_preds == 0) & (g_true == 1)).sum()
        print(f'  {group_name:20s}: {correct}/{n} ({100*correct/n:.1f}% acc) | FN (perdidos): {fn}')
    else:
        fp = ((g_preds == 1) & (g_true == 0)).sum()
        print(f'  {group_name:20s}: {correct}/{n} ({100*correct/n:.1f}% acc) | FP (pixelados error): {fp}')

# ── Curva Precision-Recall ─────────────────────────────────────────────────────
prec_c, rec_c, thr_c = precision_recall_curve(val_l, probs)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rec_c, prec_c, color='red', lw=2)
axes[0].axvline(x=0.80, ls='--', color='gray', label='Recall objetivo 80%')
axes[0].axhline(y=0.50, ls=':', color='orange', label='Precision minima 50%')
axes[0].scatter([fbeta_score(val_l, (probs >= best_thr).astype(int), beta=1, pos_label=1)],
                [classification_report(val_l, preds_best, output_dict=True, zero_division=0)['1']['precision']],
                color='blue', s=100, zorder=5, label=f'Umbral optimo {best_thr:.3f}')
axes[0].set_xlabel('Recall (menores)')
axes[0].set_ylabel('Precision (menores)')
axes[0].set_title('Curva Precision-Recall para clase MENOR')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Histograma de probabilidades por clase
minor_probs = probs[val_l == 1]
adult_probs = probs[val_l == 0]
axes[1].hist(adult_probs, bins=40, alpha=0.6, color='#3498db', label='Adultos')
axes[1].hist(minor_probs, bins=40, alpha=0.6, color='#e74c3c', label='Menores')
axes[1].axvline(x=best_thr, color='black', ls='--', lw=2, label=f'Umbral {best_thr:.3f}')
axes[1].set_xlabel('Probabilidad de ser menor')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribucion de probabilidades por clase')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\n>>> Ajusta MINOR_PROB_THRESHOLD={best_thr:.2f} en docker-compose.yml (servicio age-detection)')

## Paso 8 — Descargar modelo

In [ ]:
from google.colab import files
files.download(str(MODEL_PATH))
print(f'Descargando: {MODEL_PATH}')
print('Guardalo en: age-detection/model/age_classifier.keras')
print(f'Recuerda ajustar MINOR_PROB_THRESHOLD en docker-compose.yml')